# 🚀 DeepSeek Vulnerability Detection Training on Colab

Fine-tune DeepSeek Coder 1.3B on CVE vulnerability dataset

**Requirements:**
- Google Colab with GPU (T4 or better)
- Runtime: GPU (Runtime > Change runtime type > GPU)

**Training Time:**
- 10k samples: ~1-2 hours on T4
- Full dataset (115k): ~10-15 hours on T4

## 1️⃣ Setup Environment

In [ ]:
## 2️⃣ Upload Datasets

**Upload BOTH files using the folder icon on the left:**
1. `train-00000-of-00001.parquet` - CVE vulnerability dataset (115k samples)
2. `training_data.jsonl` - Synthetic vulnerability examples (5 samples)

The training will combine both datasets for better results!

In [ ]:
# Dataset paths - Upload both files!\nPARQUET_DATASET_PATH = "/content/train-00000-of-00001.parquet"  # CVE dataset (115k samples)\nJSONL_DATASET_PATH = "/content/training_data.jsonl"  # Synthetic vulnerability examples\n\n# Dataset options\nUSE_PARQUET = True   # Use CVE parquet dataset\nUSE_JSONL = True     # Use synthetic vulnerability examples  \nCOMBINE_DATASETS = True  # Combine both datasets for better training\n\n# Option: Mount Google Drive (uncomment if using Drive)\n# from google.colab import drive\n# drive.mount('/content/drive')\n# PARQUET_DATASET_PATH = "/content/drive/MyDrive/train-00000-of-00001.parquet"\n# JSONL_DATASET_PATH = "/content/drive/MyDrive/training_data.jsonl"

## 2️⃣ Upload Dataset

Upload your `train-00000-of-00001.parquet` file using the file upload button on the left sidebar, or use Google Drive.

In [ ]:
# Option 1: Upload file directly (click the folder icon on left, then upload)
DATASET_PATH = "/content/train-00000-of-00001.parquet"

# Option 2: Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/train-00000-of-00001.parquet"

## 3️⃣ Import Libraries

In [ ]:
import os
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## 4️⃣ Configuration

In [ ]:
# Model and training configuration
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
OUTPUT_DIR = "./deepseek-vulnerability-finetuned"
MAX_LENGTH = 512

# Training samples - adjust based on your needs and time
# For testing: 1000-5000
# For quick training: 10000-20000
# For full training: None (uses all 115k samples)
MAX_SAMPLES = 10000  # Set to None for full dataset

print(f"📊 Will train on: {MAX_SAMPLES if MAX_SAMPLES else 'ALL'} samples")

## 5️⃣ CWE Mapping

In [ ]:
# CWE ID to name mapping
CWE_NAMES = {
    89: "SQL Injection",
    79: "Cross-site Scripting (XSS)",
    78: "OS Command Injection",
    22: "Path Traversal",
    352: "Cross-Site Request Forgery (CSRF)",
    434: "Unrestricted Upload of File with Dangerous Type",
    94: "Code Injection",
    20: "Improper Input Validation",
    476: "NULL Pointer Dereference",
    770: "Allocation of Resources Without Limits",
    532: "Insertion of Sensitive Information into Log File",
    119: "Buffer Overflow",
    125: "Out-of-bounds Read",
    787: "Out-of-bounds Write",
    416: "Use After Free",
    190: "Integer Overflow",
    200: "Exposure of Sensitive Information",
    287: "Improper Authentication",
    862: "Missing Authorization",
    863: "Incorrect Authorization"
}

def get_cwe_name(cwe_id):
    """Get CWE name from ID"""
    try:
        cwe_int = int(cwe_id)
        return CWE_NAMES.get(cwe_int, f"CWE-{cwe_int}")
    except:
        return f"CWE-{cwe_id}"

## 6️⃣ Dataset Loading Functions

In [ ]:
import json

def load_jsonl_dataset(jsonl_path):
    """Load and prepare the JSONL chat dataset"""
    print(f"📊 Loading JSONL dataset from {jsonl_path}...")
    training_data = []
    
    try:
        with open(jsonl_path, 'r') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line)
                    messages = data.get('messages', [])
                    
                    system_msg = ""
                    user_msg = ""
                    assistant_msg = ""
                    
                    for msg in messages:
                        if msg['role'] == 'system':
                            system_msg = msg['content']
                        elif msg['role'] == 'user':
                            user_msg = msg['content']
                        elif msg['role'] == 'assistant':
                            assistant_msg = msg['content']
                    
                    prompt = f"""System: {system_msg}

User: {user_msg}

## 6️⃣ Load and Explore Dataset

In [ ]:
# Load dataset
print("📊 Loading dataset...")
df = pd.read_parquet(DATASET_PATH)

print(f"\n✅ Dataset loaded!")
print(f"Total samples: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check CWE distribution
print("Top 10 most common vulnerabilities:")
cwe_counts = df['primary_cwe'].value_counts().head(10)
for cwe, count in cwe_counts.items():
    print(f"  {get_cwe_name(cwe)}: {count} samples")

## 7️⃣ Prepare Training Data

In [ ]:
def prepare_dataset(df, max_samples=None):
    """Prepare CVE dataset for training"""
    if max_samples:
        df = df.head(max_samples)
    
    print(f"Preparing {len(df)} samples...")
    
    training_data = []
    skipped = 0
    
    for idx, row in df.iterrows():
        try:
            cve_id = row['id']
            description = row['descriptions']
            primary_cwe = row['primary_cwe']
            
            if pd.isna(primary_cwe) or pd.isna(description):
                skipped += 1
                continue
            
            cwe_name = get_cwe_name(primary_cwe)
            
            # Create instruction-following format
            prompt = f"""System: You are a security expert analyzing vulnerabilities.

User: Analyze this vulnerability:

CVE ID: {cve_id}
Description: {description}

What type of vulnerability is this and what are the security implications?

Assistant: This is a {cwe_name} vulnerability (CWE-{primary_cwe}). {description}

Security Implications:
- This type of vulnerability can lead to unauthorized access, data breaches, or system compromise.
- Attackers can exploit this weakness to bypass security controls.
- Proper input validation and security best practices should be implemented to mitigate this risk."""
            
            training_data.append({"text": prompt})
            
            if (idx + 1) % 1000 == 0:
                print(f"  Processed {idx + 1} samples...")
                
        except Exception as e:
            skipped += 1
            continue
    
    print(f"\n✅ Prepared {len(training_data)} training samples")
    print(f"⚠️  Skipped {skipped} samples (missing data)")
    
    return Dataset.from_list(training_data)

# Prepare dataset
dataset = prepare_dataset(df, max_samples=MAX_SAMPLES)

In [ ]:
# Split into train/validation
print("\n📊 Splitting dataset...")
train_test_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Eval samples: {len(eval_dataset)}")

# Show example
print("\n📝 Example training sample:")
print("="*70)
print(train_dataset[0]['text'][:500] + "...")
print("="*70)

## 8️⃣ Load Tokenizer and Model

In [ ]:
# Load tokenizer
print("📝 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Tokenizer loaded!")

In [ ]:
# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

print("🔤 Tokenizing datasets...")
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print("✅ Tokenization complete!")

In [ ]:
# Load model
print("🤖 Loading base model...")
print("This may take a few minutes...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Model loaded!")

## 9️⃣ Configure LoRA

In [ ]:
# Configure LoRA for efficient fine-tuning
print("⚙️ Configuring LoRA...")

lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # LoRA alpha
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Show trainable parameters
print("\n📊 Model parameters:")
model.print_trainable_parameters()

## 🔟 Training Configuration

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    warmup_steps=100,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✅ Training configuration ready!")

## 1️⃣1️⃣ Initialize Trainer

In [ ]:
# Initialize trainer
print("🏋️ Initializing trainer...")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✅ Trainer ready!")

## 1️⃣2️⃣ Start Training! 🔥

This will take a while. You can monitor progress below.

**Estimated time:**
- 10k samples: 1-2 hours
- 50k samples: 5-7 hours
- 115k samples: 10-15 hours

In [ ]:
# Start training
print("="*70)
print("🔥 STARTING TRAINING")
print("="*70)
print("\n⏰ This will take a while. Don't close this tab!\n")

trainer.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

## 1️⃣3️⃣ Save Model

In [ ]:
# Save final model
print("💾 Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ Model saved to: {OUTPUT_DIR}")
print("\n📦 Download the model folder to use it locally!")

## 1️⃣4️⃣ Test the Model

In [ ]:
# Test function
def analyze_vulnerability(cve_id, description):
    """Test the trained model"""
    prompt = f"""System: You are a security expert analyzing vulnerabilities.

User: Analyze this vulnerability:

CVE ID: {cve_id}
Description: {description}

What type of vulnerability is this and what are the security implications?

Assistant:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Assistant:")[-1].strip()

# Test examples
print("="*70)
print("🧪 TESTING MODEL")
print("="*70)

test_cases = [
    ("CVE-2019-3494", "Simply-Blog through 2019-01-01 has SQL Injection via the admin/deleteCategories.php delete parameter."),
    ("CVE-TEST-XSS", "Web application allows user input to be rendered without sanitization in HTML output."),
    ("CVE-TEST-CMD", "Application executes system commands with unsanitized user input.")
]

for cve_id, desc in test_cases:
    print(f"\n{'='*70}")
    print(f"CVE: {cve_id}")
    print(f"Description: {desc}")
    print(f"\nAnalysis:")
    print(analyze_vulnerability(cve_id, desc))

print(f"\n{'='*70}")
print("✅ Testing complete!")
print("="*70)

## 1️⃣5️⃣ Download Model

To download the trained model:
1. Click the folder icon on the left
2. Navigate to `deepseek-vulnerability-finetuned`
3. Right-click the folder and select "Download"

Or zip it first:

In [ ]:
# Zip the model for easier download
!zip -r deepseek-vulnerability-finetuned.zip deepseek-vulnerability-finetuned/

print("\n✅ Model zipped!")
print("📦 Download: deepseek-vulnerability-finetuned.zip")
print("\nFile size:")
!ls -lh deepseek-vulnerability-finetuned.zip

## 1️⃣6️⃣ Optional: Upload to Hugging Face Hub

Share your model on Hugging Face!

In [ ]:
# Uncomment and run to upload to Hugging Face
# !pip install -q huggingface_hub
# from huggingface_hub import notebook_login
# notebook_login()

# # Push to hub
# model.push_to_hub("your-username/deepseek-vulnerability-detector")
# tokenizer.push_to_hub("your-username/deepseek-vulnerability-detector")

---

## 🎉 Training Complete!

Your model is now trained and ready to use for vulnerability detection.

**Next steps:**
1. Download the model
2. Test it locally with your code
3. Deploy for production use

**To use locally:**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-1.3b-instruct")
model = PeftModel.from_pretrained(base_model, "./deepseek-vulnerability-finetuned")
tokenizer = AutoTokenizer.from_pretrained("./deepseek-vulnerability-finetuned")
```